In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 6** - Cake Recipie Optimisation

- This function is about optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk.
  - Each recipe is evaluated with a combined score based on _flavour_, _consistency_, _calories_, _waste_ and _cost_, where each factor contributes **negative points** as judged by an expert taster. This means the total score is negative by design.

- **Goal** - maximisation problem, your goal is to _bring that score as close to zero_ as possible or, equivalently, to maximise the negative of the total sum.

- **Input** - 5D (20,5)
- **Output** - 5D (20,)
- **Goal** - maximisation

In [2]:
X = np.load('/Users/pratham/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_inputs.npy')
Y = np.load('/Users/pratham/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_outputs.npy')

In [3]:
# New data from Week 3 (Function 6)
X_w3_new_point = np.array([0.727708, 0.335856, 0.785846, 0.777106, 0.230214], dtype=np.float64)
Y_w3_new_point = np.array([-0.6048291372506664], dtype=np.float64)

# Append the new data points
X_updated = np.vstack((X, X_w3_new_point.reshape(1, -1)))
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)
Y_updated = np.append(Y, Y_w3_new_point)[unique_indices]

# Save the updated arrays
np.save('/Users/pratham/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_inputs.npy', X_unique)
np.save('/Users/pratham/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_outputs.npy', Y_updated)

In [4]:
# Show updated arrays
print("Updated Inputs (X) - Function 5: ", X_unique)
print("Updated Outputs (Y) - Function 5: ", Y_updated)

Updated Inputs (X) - Function 5:  [[0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.12667892 0.2914703  0.06452848 0.6805146  0.89281919]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.441233   0.998122   0.001245   0.556711   0.321098  ]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.727708   0.335856   0.785846   0.777106   0.230214  ]
 [0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.77062024 0.11440374 0.04677993 0.6

### **Interpretation of the Output**

- A result of $\approx 0.60$ is significantly better than zero or negative results, but it doesn't have the "explosive" quality of a global maximum. 
    - This suggests the function might be smoother and less volatile than Function 5.
    
- I have one point in a space with $2^5 = 32$ quadrants. 
    - Mathematically, the Gaussian Process still thinks $99.9\%$ of the map is unknown.
    
- So I will need to perform Strategic Exploration. 
    - I will use the $0.60$ result as a lighthouse and search the "dark" areas around it to see if the signal increases or drops off sharply.

### **Bayesian Optimisation** - Gaussian Optimisation

- `Matern(nu=2.5)` + `WhiteKernel`.
    - The Matern kernel handles 5D "ridges" better than a standard RBF. 
        - The WhiteKernel adds a noise-buffer to prevent the model from overfitting this single $0.60$ result.

- **ARD Length-Scales ($l$)**:
    - This allows the model to determine if, for example, $x_1$ and $x_5$ are the primary drivers while the middle three variables are irrelevant.
    
- Alpha `alpha=1e-3`
    - Provides numerical stability for the matrix inversion in high-dimensional space.

In [5]:
kernel = Matern(
    length_scale=[0.2]*5,
    nu=2.5) \
         + WhiteKernel(noise_level=1e-4)

model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-3, 
    n_restarts_optimizer=35,
    normalize_y=True
)
model.fit(X_unique, Y_updated)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",Matern(length..._level=0.0001)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",0.001
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",35
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`G

### **Acquisition Function** - Upper Confidence Bound (UCB)

- UCB with $\kappa = 2.576$
    - In 5D, Expected Improvement (EI) is often "blind" because the probability of improving on a sparse map is mathematically near zero. UCB is an explorer. 
    - It uses the uncertainty ($\sigma$) as a fuel source to push into the furthest, darkest corners of the 5D hypercube.
    
- Theory: A $\kappa$ of $2.576$ targets the $99\%$ confidence interval.

In [6]:
def upper_confidence_bound(X_grid, model, kappa=2.576):
    mu, sigma = model.predict(X_grid, return_std=True)
    return mu + kappa * sigma

# Generate 150,000 random points (Monte Carlo) to cover 5D space
x_grid = np.random.uniform(0, 1, size=(150000, 5))
ucb_vals = upper_confidence_bound(x_grid, model, kappa=2.576)
next_query = x_grid[np.argmax(ucb_vals)]

print(f"Strategic Week 3 Query (Function 6): {next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}")

Strategic Week 3 Query (Function 6): 0.461344-0.323423-0.803851-0.969268-0.025109


If the next result is significantly lower (e.g., $0.05$), it means the $0.60$ was a local feature or a very narrow ridge. In that case, I will need to tighten the length-scales to $0.05$ to search around the successful coordinate. For now, trust the UCB to find the edge of the 0.60 plateau.